In [1]:
import numpy as np
from utils import ResultsManager
from scipy.stats import mannwhitneyu
from scipy.stats import bootstrap

In [ ]:
managers = {
    "hcgp": ResultsManager('results/lsmgrid_experiment_hcgp_aug10.json'),
    "ga"  : ResultsManager('results/lsmgrid_experiment_ga_aug10.json'),
    "csa" : ResultsManager('results/lsmgrid_experiment_csa_aug10.json')
}

In [3]:
dataset_printable = {
    'FSDD':"FSDD",
    'NTIDIGITS':'N-TIDIGITS',
    'NMNIST':'N-MNIST',
    'MITBIH':'MIT-BIH'   
}

readout_printable = {
    'LOGREG':"\\gls{lr}",
    'RF': "\\gls{rf}",
    'SVM': "\\gls{svm}",
    'KNN': "\\gls{knn}"
}

def count_synapses(grid):
    suma = 0
    for cell in grid:
        if not 'd3_lims' in cell:
            start = int(cell['d2_lims'][0])
            end  = int(cell['d2_lims'][1]) 
            suma += end-start
        else:
            start_d2 = int(cell['d2_lims'][0])
            end_d2   = int(cell['d2_lims'][1])
            start_d3 = int(cell['d3_lims'][0])
            end_d3   = int(cell['d3_lims'][1])
            suma += (end_d2-start_d2)*(end_d3-start_d3)
    return suma

In [22]:
for dataset in ['FSDD','NTIDIGITS','NMNIST','MITBIH']:

	print("       \\multicolumn{12}{c}{\\textbf{", dataset_printable[dataset], "}} \\\\ \\hline")
	for readout in ['LOGREG','RF','SVM','KNN']:
		for algo in ['hcgp','ga','csa']:
			trials = managers[algo].results[f'{dataset}+{readout}']

			# Seccion accuracies
			accs      = np.array([ trial['final_acc'] for trial in trials ])
			accs_mean = accs.mean().round(3)
			accs_min  = accs.min().round(3)
			accs_max  = accs.max().round(3)
			accs_std  = accs.std().round(3)

			boots = bootstrap((accs,), np.mean, confidence_level=0.95, n_resamples=10000, method="percentile")
			low_acc  = boots.confidence_interval[0].round(3)
			high_acc = boots.confidence_interval[1].round(3)


			# Seccion neurons
			neu   	 = np.array([ len(trial['best_grid']) for trial in trials ])
			neu_mean = neu.mean().round().astype(np.int64)
			boots 	 = bootstrap((neu,), np.mean, confidence_level=0.95, n_resamples=10000, method="percentile")
			low_neu  = boots.confidence_interval[0].round().astype(np.int64)
			high_neu = boots.confidence_interval[1].round().astype(np.int64)

			# Seccion sinapsis
			syn      = np.array([ count_synapses(trial['best_grid']) for trial in trials ])
			syn_mean = syn.mean().round().astype(np.int64)
			boots    = bootstrap((syn,), np.mean, confidence_level=0.95, n_resamples=10000, method="percentile")
			low_syn  = boots.confidence_interval[0].round().astype(np.int64)
			high_syn = boots.confidence_interval[1].round().astype(np.int64)
			
			# Print
			print_readout = readout_printable[readout] if algo=='ga' else ""
			print_line    = "\\hline" if algo=='csa' else ""

			row =  f" {print_readout:<10} & {algo} & "
			
			row += f"{accs_min:.3f} & "
			row += f"{accs_max:.3f} & "
			row += f"{accs_mean:.3f} [{low_acc:.3f}-{high_acc:.3f}] & "
			row += f"{accs_std:.3f} &"

			row += f"{neu.min()} & "
			row += f"{neu.max()} & "
			row += f"{neu_mean} [{low_neu}-{high_neu}] & "

			row += f"{syn.min()} & "
			row += f"{syn.max()} & "
			row += f"{syn_mean:>3} [{low_syn}-{high_syn}] "


			row += f" \\\\ {print_line}"
			print(row)


       \multicolumn{12}{c}{\textbf{ FSDD }} \\ \hline
            & hcgp & 0.848 & 0.935 & 0.900 [0.894-0.907] & 0.018 &29 & 51 & 40 [38-42] & 71 & 200 & 123 [112-135]  \\ 
 \gls{lr}   & ga & 0.887 & 0.933 & 0.914 [0.910-0.917] & 0.009 &27 & 36 & 33 [32-34] & 59 & 164 &  97 [88-107]  \\ 
            & csa & 0.853 & 0.920 & 0.888 [0.882-0.893] & 0.015 &29 & 36 & 33 [32-33] & 61 & 255 & 134 [119-152]  \\ \hline
            & hcgp & 0.918 & 0.958 & 0.940 [0.936-0.943] & 0.009 &18 & 29 & 23 [22-24] & 52 & 149 &  87 [79-96]  \\ 
 \gls{rf}   & ga & 0.928 & 0.962 & 0.947 [0.944-0.950] & 0.009 &28 & 36 & 32 [32-33] & 57 & 191 & 114 [102-127]  \\ 
            & csa & 0.915 & 0.960 & 0.936 [0.932-0.939] & 0.011 &27 & 35 & 32 [31-33] & 76 & 272 & 130 [113-149]  \\ \hline
            & hcgp & 0.948 & 0.978 & 0.962 [0.959-0.965] & 0.008 &27 & 44 & 38 [36-40] & 58 & 147 &  94 [88-101]  \\ 
 \gls{svm}  & ga & 0.937 & 0.970 & 0.957 [0.954-0.960] & 0.008 &30 & 36 & 34 [33-34] & 71 & 206 & 127 [115-139]